# Ga-Ru Adjustment Workflow

This notebook demonstrates how the `gliquid` workflow can be used to adjust predictions using experimental measurements:

1. Generate an initial ML-guided prediction.
2. Overlay measured DSC points.
3. Refine liquid parameters using the measured liquidus constraints.
4. Compare original vs revised phase diagram and Gibbs-energy curves.

In [ ]:
import os
import gliquid.config as cfg
from gliquid.production_model_runner import ProductionModelRunner

# Required for fetching any uncached MP data.
os.environ["NEW_MP_API_KEY"] = "YOUR_API_KEY_HERE"

# Bundle with trained production models + feature metadata.
runner = ProductionModelRunner(str(cfg.data_dir) + "/20260217_135723")

## 1 - Plot predicted liquidus curve

Use latest ML model ensemble to predict non-ideal mixing parameters for the Ga-Ru system and plot the resulting phase diagram.

In [ ]:
from gliquid.binary import BinaryLiquid, BLPlotter

system = "Ga-Ru"
bl = BinaryLiquid.from_cache(system, param_format='comb-exp')

pred_params = runner.predict_system(system) + [0]
bl.update_params(pred_params)
predicted_liquidus_line = [[p[0], p[1]] for p in bl.phases[-1]['points']]  # Keep a copy for later overlay

blp = BLPlotter(bl)
blp.show('pred')   # Predicted phase diagram
blp.show('ch+g')   # DFT convex hull with liquid free-energy overlay

Following this initial prediction, this system was identified as a system that our collaborators at Baylor University
were capable of conducting an assessment on using their DSC techniques.

Below is a table of measurements from their assessment of the Ru-Ga system:

## Table 1. DSC runs of the Ru-Ga system and weight % change

| Composition (at.% Ru: at.% Ga) | Solidus (±0.5 °C) | Liquidus (±0.5 °C) | Weight Change (±0.1 %) |
|-------------------------|-------------------|-------------------|------------------------|
| 5: 95                   | -                 | 1008.8            | 0.149                  |
| 15: 85a                 | 1095.5            | 1103.7            | 0.034                  |
| 15: 85b                 | 1085.9            | 1111.1            | 0.132                  |
| 15: 85c                 | -                 | 1109.6            | 0.004                  |
| 20: 80                  | 1087.6            | -                 | 0.065                  |
| 25: 75a                 | 1090.3            | -                 | 0.022                  |
| 25: 75b                 | 1090.4            | -                 | 0.086                  |
| 25: 75c                 | 1088.9            | -                 | 0.056                  |
| 30: 70                  | 1090.9            | -                 | 0.009                  |
| 35: 65a                 | -                 | -                 | 0.027                  |
| 35: 65b                 | -                 | -                 | 0.219                  |
| 35: 65c                 | -                 | -                 | 0.002                  |
| 45: 55a                 | -                 | -                 | 0.014                  |
| 45: 55b<sup>a</sup>     | -                 | -                 | 0.143                  |
| 45: 55c<sup>b</sup>     | -                 | -                 | 0.006                  |
| 50: 50                  | -                 | -                 | 0.101                  |
| 55: 45a<sup>c</sup>     | -                 | -                 | 0.028                  |
| 55: 45b<sup>d</sup>     | -                 | -                 | 0.015                  |
| 60: 40                  | -                 | -                 | 0.150                  |
| 68: 32                  | -                 | -                 | 0.008                  |

The letters a,b,c after the compositions are there only to distinguish samples of the same composition. The exponents are to signify samples that were prepared differently while trying to determine the preparation method that provides the most homogenous sample.<br> 
<sup>a</sup> Sample used larger Ga pieces relative to the fine Ga powder used for other samples<br>
<sup>b</sup> Sample used Ru pieces that were wet with molten Ga<br>
<sup>c, d</sup> Sample used Ru powder that was pressed into a pellet, then Ga pieces were dropped on top<br>

Credit:

Mario A. Plata,<sup>a</sup> J. Streit Smith,<sup>a</sup> and Julia Y. Chan<sup>* a</sup><br>
<sup>a</sup> Department of Chemistry and Biochemistry, Baylor University, Waco, Texas 76798, United States.<br>
<sup>*</sup> Department of Chemistry and Biochemistry, Baylor University, Waco, Texas 76798, United States;<br>
https://orcid.org/0000-0003-4434-2160; Email: Julia_Chan@baylor.edu<br>

In [ ]:
# composition at% Ru, solidus temperature (C), liquidus temperature (C)
measured_points_wtpct = [[5,    None,   1008.8],
                         [15,   1095.5, 1103.7],
                         [15,   1085.9, 1111.1],
                         [15,   None,   1109.6],
                         [20,   1087.6, None],
                         [25,   1090.3, None],
                         [25,   1090.4, None],
                         [25,   1088.9, None],
                         [30,   1090.9, None],
                         [35,   None,   None],
                         [35,   None,   None],
                         [35,   None,   None],
                         [45,   None,   None],
                         [45,   None,   None],
                         [45,   None,   None],
                         [50,   None,   None],
                         [55,   None,   None],
                         [55,   None,   None],
                         [60,   None,   None],
                         [68,   None,   None]] 

## 2 - Process measured DSC points

Convert repeated measurements into averaged solidus/liquidus observations at each composition.

In [ ]:
def process_measured_points(measured_points_wtpct):
    """
    Average multiple measurements at the same composition.
    Return solidus points, liquidus points, and compositions with no melting.
    """
    s, l, no_melt = [], [], []
    s_dict, l_dict = {}, {}

    for wt_pct, solidus, liquidus in measured_points_wtpct:

        if solidus is None and liquidus is None and wt_pct not in no_melt:
            no_melt.append(wt_pct)
        if solidus is not None:
            if wt_pct not in s_dict:
                s.append([wt_pct, 0])
            s_dict[wt_pct] = s_dict.get(wt_pct, []) + [solidus]
        if liquidus is not None:
            if wt_pct not in l_dict:
                l.append([wt_pct, 0])
            l_dict[wt_pct] = l_dict.get(wt_pct, []) + [liquidus]

    for point in s:
        point[1] = round(sum(s_dict[point[0]]) / len(s_dict[point[0]]), 1)
    for point in l:
        point[1] = round(sum(l_dict[point[0]]) / len(l_dict[point[0]]), 1)

    return s, l, no_melt

solidus_points, liquidus_points, no_liquid_comps = process_measured_points(measured_points_wtpct)
print("Solidus points:", solidus_points)
print("Liquidus points:", liquidus_points)
print("Compositions with no melting (at. %):", no_liquid_comps)


## 3 - Overlay measurements on the baseline prediction

Plot the measured points against the initial predicted phase diagram.

In [ ]:
import plotly.graph_objects as go

def add_experimental_traces(figure: go.Figure, inc_no_liquid=True):
    """Add measured observations to a Plotly phase-diagram figure."""
    
    if inc_no_liquid:
        # Vertical guides for compositions where no liquid phase was observed
        for i, x_val in enumerate(no_liquid_comps):
            figure.add_trace(go.Scatter(
                x=[x_val, x_val],
                y=[0, 900],
                mode='lines',
                line=dict(color='red', width=2),
                name='No Liquid Phase Observed' if i == 0 else None,
                showlegend=i == 0,
            ))
            figure.add_trace(go.Scatter(
                x=[x_val, x_val],
                y=[900, 1200],
                mode='lines',
                line=dict(color='red', width=2, dash='dash'),
                showlegend=False
            ))

    figure.add_trace(go.Scatter(
            x=[point[0] for point in solidus_points][1:],
            y=[point[1] for point in solidus_points][1:],
            mode='markers',
            marker=dict(color='#ADEBB3', size=14, symbol='triangle-up', line=dict(width=1, color='black')),
            name='Solidus Observed'
    ))

    figure.add_trace(go.Scatter(
            x=[point[0] for point in liquidus_points],
            y=[point[1] for point in liquidus_points],
            mode='markers',
            marker=dict(color='cornflowerblue', size=12, symbol='square', line=dict(width=1, color='black')),
            name='Liquidus Observed'
    ))

fig = blp.get_plot('pred')
add_experimental_traces(fig)
fig.update_layout(yaxis_range=[-150, 2550])
fig.show()

## 4 - Refine liquid parameters using measured constraints

Use measured liquidus information to adjust the non-ideal mixing parameters.

In [ ]:
import numpy as np
from gliquid.binary import _x_vals

def optimize_predicted_params(bl: BinaryLiquid, known_points: list, predicted_params: list) -> tuple:

    hull_points = np.array([[p['comp'], p['enthalpy']] for p in bl.phases if 'comp' in p])
    bl.eqs['h_hull_interp'] = np.interp(_x_vals[1:-1], hull_points[:, 0], hull_points[:, 1])
    bl.comp_range_fit_lim = 0
    bl.init_error = False
    bl.max_liq_temp = max(bl.component_data.values(), key=lambda x: x[1])[1]
    bl.min_liq_temp = min(bl.component_data.values(), key=lambda x: x[1])[1]
    bl.digitized_liq = [[p[0], p[1]] for p in known_points]

    mae, _, _, _ = bl.calculate_deviation_metrics(ignored_ranges=False, allow_sparse_data=True)
    print(f"Known points (Kelvin): {known_points}")
    print(f"Liquidus MAE from known points: {int(mae)}K")

    print(f"Original predicted parameters: {predicted_params}")

    bl.fit_parameters(verbose=True, max_iter=128, check_phase_mismatch=False, allow_sparse_data=True, 
                      disable_inv_constrs=True, params_init=predicted_params)
    
    print(f"Adjusted parameters: {bl.get_params()}")
    return bl.get_params()


GaRu_phase = {'name': 'GaRu', 'comp': 0.5, 'points': [], 'enthalpy': -0.301 * 96485, 'entropy': 10.2}
if not any(phase['name'] == 'GaRu' for phase in bl.phases):
    bl.phases.insert(3, GaRu_phase)
bl.phases[1]['entropy'] = -1.4

print("Solid phase energies (eV/atom):")
for p in bl.phases:
    if 'energy' in p:
        print(p['name'], round(p['energy']/96485, 3))
print("-" * 30)

# Reformat measured liquidus points to [at. fraction, Kelvin]
measured_liq_points = [[p[0]/100.0, p[1]+273.15] for p in liquidus_points]
measured_liq_points.append([0.73, 1125+273.15]) # Add suspected pre-melting point to stabilize optimization
adj_params = optimize_predicted_params(bl, measured_liq_points, pred_params)
bl.temp_range = [253, 1000]
bl.update_phase_points()
blp = BLPlotter(bl)
fig = blp.get_plot('pred')
add_experimental_traces(fig)
fig.show()

## 5 - Compare original vs revised predictions

Contrast the updated phase diagram against the original ML prediction.

In [ ]:
# no_liquid_comps.remove(50)
bl.update_params(adj_params) # update the parameters and generate the DFT-referenced phase diagram
bl.digitized_liq = predicted_liquidus_line # set the original predicted line as the liquidus reference for plotting
fig = blp.get_plot('pred+liq') # generate the plot as a figure object so it can be modified
add_experimental_traces(fig, inc_no_liquid=True) # add the experimental data to the plot

# Manually edit trace colors and names and update layout. This may need to be altered if the trace order changes
fig.data[0].line.color = '#117733' # reference liquidus - plotted line 
fig.data[7].line.color = '#962EFF' # adjusted liquidus - plotted line
fig.data[8].line.color = '#117733' # reference liquidus - legend
fig.data[8].name = 'Original Liquidus Prediction'
fig.data[9].line.color = '#962EFF' # adjusted liquidus - legend
fig.data[9].name = 'Revised Liquidus Prediction'
fig.update_layout(width=900, height=700)
fig.show()

# Create another version without labels for manual labeling in powerpoint
# fig_dict = fig.to_dict()
# fig_dict['layout'].pop('title', None)
# fig_dict['layout']['xaxis'].pop('title', None)
# fig_dict['layout']['yaxis'].pop('title', None)
# fig_dict['layout'].pop('annotations', None)
# fig_no_labels = go.Figure(fig_dict)
# fig_no_labels.layout.showlegend = False
# fig_no_labels.layout.xaxis.showticklabels = False
# fig_no_labels.layout.yaxis.showticklabels = False
# fig_no_labels.update_layout(yaxis_range=[-150, 2550])
# fig_no_labels.add_trace(go.Scatter(
#     x=[bl.phases[0]['comp']*100, bl.phases[1]['comp']*100],
#     y=[bl.component_data[bl.components[0]][1]-275, bl.component_data[bl.components[0]][1]-275],
#     mode='lines',
#     line=dict(color='silver', width=4),
#     showlegend=False))
# fig_no_labels.show()

# Save the figure without labels
# figures_dir = cfg.project_root / "figures"
# if os.path.isdir(figures_dir):
#     fig_no_labels.write_image(figures_dir / "Ga-Ru_phase_diagram_cleaned.svg")
#     print(f"Figure saved to {figures_dir}")

In [ ]:
from pymatgen.analysis.phase_diagram import PDPlotter, PhaseDiagram, PDEntry  # The PMG PDPlotter source code is modified here
pdp = PDPlotter(bl.dft_ch)
from gliquid.binary import t_sym, a_sym, b_sym, c_sym, d_sym, xb_sym
import sympy as sp

t_eval_c = 1108
t_eval = t_eval_c + 273.15

def get_g_curve(A=0, B=0, C=0, D=0, T=0, name="") -> go.Scatter:
    """
    Args:
        A, B, C, D (float): Non-ideal mixing parameters.
        T (float): Temperature in Kelvin.

    Returns:
        go.Scatter: Plotly scatter trace for the Gibbs free energy curve.
    """
    g = bl.eqs['g_liquid'].subs({t_sym: T, a_sym: A, b_sym: B, c_sym: C, d_sym: D})
    gliq_vals = sp.lambdify(xb_sym, g, 'numpy')(_x_vals[1:-1]) if g.has(xb_sym) else [0] * len(_x_vals[1:-1])
    ga = np.float64(bl.eqs['ga'].subs({t_sym: T}) / 96485)
    gb = np.float64(bl.eqs['gb'].subs({t_sym: T}) / 96485)
    if name == "":
        name += "Liquid T=" + str(int(T-273.15)) + "C"

    return go.Scatter(
        x=_x_vals,
        y=[ga] + [g / 96485 for g in gliq_vals] + [gb],
        mode='lines',
        name=name
    )

fig = pdp.get_plot()
fig.add_trace(get_g_curve(A=pred_params[0], B=pred_params[1], C=pred_params[2], D=pred_params[3],
                          T=t_eval, name=f"Original T={t_eval_c}C").update(line=dict(color='#117733', width=3)))

print("Phase energies at t_eval (eV/atom):")
for p in bl.phases:
    if 'enthalpy' in p:
        print(p['name'] + f" @ t_eval ({t_eval})", (p['enthalpy'] - p.get('entropy', 0)*t_eval)/96485.0)

allpoints = [[p['comp'], (p['enthalpy'] - p.get('entropy', 0)*t_eval)/96485.0] for p in bl.phases[:-1]]
newpoints = [[p['comp'], (p['enthalpy'] - p.get('entropy', 0)*t_eval)/96485.0] for p in bl.phases[:-1] if 'entropy' in p]
oldpoints = [[p['comp'], (p['enthalpy'] - p.get('entropy', 0)*t_eval)/96485.0] for p in bl.phases[:-1] if 'entropy' not in p]

# Plot connecting lines between all points
for i in range(len(allpoints) - 1):
    fig.add_trace(go.Scatter(
        x=[allpoints[i][0], allpoints[i+1][0]],
        y=[allpoints[i][1], allpoints[i+1][1]],
        mode='lines',
        line=dict(color='#555555', dash='dot', width=3),
        showlegend=False
    ))

# Fix the data structure - remove the extra brackets
fig.add_trace(go.Scatter(
    x=[p[0] for p in newpoints],
    y=[p[1] for p in newpoints],
    mode='markers',
    marker=dict(color="purple", size=16, symbol='circle', line=dict(width=2, color='black')),
    showlegend=False
))

fig.add_trace(go.Scatter(
    x=[p[0] for p in oldpoints],
    y=[p[1] for p in oldpoints],
    mode='markers',
    marker=dict(color="darkgreen", size=16, symbol='circle', line=dict(width=2, color='black')),
    showlegend=False
))

# Add smaller purple circles inside the darkgreen circles
fig.add_trace(go.Scatter(
    x=[p[0] for p in oldpoints],
    y=[p[1] for p in oldpoints],
    mode='markers',
    marker=dict(color="purple", size=9, symbol='circle', line=dict(width=0.5, color='#Be03Fd')),
    showlegend=False
))

fig.add_trace(get_g_curve(A=adj_params[0], B=adj_params[1], C=adj_params[2], D=adj_params[3],
                          T=t_eval, name=f"Revised T={t_eval_c}C").update(line=dict(color='#962EFF', width=3)))

fig.update_layout(plot_bgcolor='white',
                  paper_bgcolor='white', 
                  xaxis=dict(title=dict(text="Composition (at. %)", font=dict(size=18)),
                            tickfont=dict(color='black'), mirror=False),
                  yaxis=dict(title=dict(text="", font=dict(size=18)),
                            tickfont=dict(color='black'), mirror=False),
                  font=dict(color='black', size=13),  # Sets default font color for all text elements
                  width=800, height=450)

fig.show()
# fig_dict = fig.to_dict()
# fig_dict['layout'].pop('annotations', None)
# fig_dict['layout'].pop('title', None)
# fig_dict['layout']['xaxis'].pop('title', None)
# fig_dict['layout']['yaxis'].pop('title', None)
# fig_dict['data'] = [trace for trace in fig_dict['data'] if trace.get('mode') != 'text']
# cleaned = go.Figure(fig_dict)
# cleaned.layout.showlegend = False
# cleaned.layout.xaxis.showticklabels = False
# cleaned.layout.yaxis.showticklabels = False
# cleaned.update_layout(xaxis_range=[-0.03, 1.015])
# cleaned.show()

# figures_dir = cfg.project_root / "figures"
# if os.path.isdir(figures_dir):
#     fig.write_image(figures_dir / "Ga-Ru_chull_overlaid.svg")
#     cleaned.write_image(figures_dir / "Ga-Ru_chull_overlaid_cleaned.svg")
#     print(f"Figures saved to {figures_dir}")
